# Crypto Markout Analysis: Binance BTCUSDT

Markout curves on real cryptocurrency trade data, from the **maker's perspective**.
Data: Binance spot BTCUSDT, 2026-03-01, via [data.binance.vision](https://data.binance.vision).

In [5]:
import polars as pl
from fetch_data import fetch_binance_aggtrades, prepare_binance_trades_and_quotes

import markoutlib as mo

ImportError: cannot import name 'fetch_binance_aggtrades' from 'fetch_data' (c:\Users\jgehu\QUANT\Projects\markoutlib\examples\fetch_data.py)

## Download and prepare data

In [ ]:
raw = fetch_binance_aggtrades("BTCUSDT", "2026-03-01")
trades, quotes = prepare_binance_trades_and_quotes(raw)

print(f"Trades: {trades.shape[0]:,}")
print(f"Quotes: {quotes.shape[0]:,}")
print(f"Time range: {trades['timestamp'].min()} to {trades['timestamp'].max()}")

## Compute markouts

Wall-clock horizons from 1 second to 5 minutes, viewed from the
maker's perspective: positive values mean the maker profited.

In [ ]:
result = mo.compute(
    trades=trades,
    quotes=quotes,
    horizons=mo.seconds(1, 5, 10, 30, 60, 300),
    unit="bps",
    perspective="maker",
)

## Markout decay curve

The curve shows how much a market maker loses, on average, after
filling a trade. A deeper negative is worse for the maker.

In [ ]:
result.plot.curve()

The maker loses about 1.4 bps within the first second and the loss deepens to roughly 2 bps by 60 seconds. The decay half-life is under a second — the bulk of the adverse selection hits almost immediately. The slight recovery at 300s (from -2.1 back to -1.9 bps) suggests a transient overshoot before the mid settles.

## Segment by trade size

Bucket trades into small / medium / large by quantity. From the maker's
perspective, which trades hurt the most?

In [ ]:
q33 = trades["size"].quantile(0.33)
q67 = trades["size"].quantile(0.67)

trades_tagged = trades.with_columns(
    pl.when(pl.col("size") <= q33)
    .then(pl.lit("small"))
    .when(pl.col("size") <= q67)
    .then(pl.lit("medium"))
    .otherwise(pl.lit("large"))
    .alias("size_bucket")
)

result_sized = mo.compute(
    trades=trades_tagged,
    quotes=quotes,
    horizons=mo.seconds(1, 5, 10, 30, 60, 300),
    unit="bps",
    perspective="maker",
)

result_sized.plot.curve(by="size_bucket")

Small trades are the most toxic to the maker at every horizon (-1.5 bps at 1s, deepening to -2.2 at 5 min), while medium trades are the least costly (-1.2 bps at 1s). This is characteristic of crypto spot: informed flow arrives in small, frequent orders to minimise market impact, not in large blocks.

## Weighted vs unweighted comparison

Size-weighting gives each trade influence proportional to its notional.
If larger trades carry more directional information per unit, the
weighted markout will be more negative than the unweighted.

In [ ]:
result.compare(weight="size")

The weighted markout is substantially worse for the maker at every horizon (-1.7 vs -1.4 bps at 1s, -2.1 vs -1.9 at 5 min). Despite small trades being more toxic per trade, larger trades carry disproportionate directional information per unit of notional — so the dollar-weighted cost of adverse selection is higher than the trade-count average suggests.

## Distribution at 30 seconds

In [ ]:
result.plot.distribution(horizon=mo.seconds(30))